# CUDA N-body Simulation: Google Colab Runner
**Owner:** Shadhin  
**Course:** CSE-402 Numerical Project  

This notebook compiles and benchmarks both the **CUDA GPU implementation** and the **Scalar CPU Baseline** on Google Colab (Tesla T4 / P100).

> **Prerequisite:** Make sure your Colab runtime is set to GPU:  
> `Runtime` → `Change runtime type` → `T4 GPU`.

## 1. Inspect Hardware & Environment

In [ ]:
!nvidia-smi
!nvcc --version
!lscpu | grep "Model name\|CPU(s):"

## 2. Clone Repository & Setup Branch

In [ ]:
import os
if not os.path.exists("CSE-402-Nuremical-Project"):
    !git clone https://github.com/rageeb-hasan-shafee/CSE-402-Nuremical-Project.git
%cd CSE-402-Nuremical-Project
!git checkout shadhin_dev || git checkout -b shadhin_dev origin/shadhin_dev || true
!git status

## 3. Generate Initial Conditions (IC CSVs)

In [ ]:
!python common/python/export_ic.py --all

## 4. Build Binaries
1. **Scalar CPU baseline (`nbody_serial`)**: compiled with `g++ -O3 -std=c++17`
2. **CUDA GPU binary (`nbody_cuda`)**: compiled with `nvcc -O3 -std=c++17 -arch=sm_75`

In [ ]:
# 1. Compile Scalar CPU baseline
!g++ -O3 -std=c++17 -Icommon/cpp backends/serial/nbody_serial.cpp -o backends/serial/nbody_serial

# 2. Compile CUDA GPU binary
%cd backends/cuda
!nvcc -O3 -std=c++17 -arch=sm_75 -I../../common/cpp nbody_cuda.cu -o nbody_cuda
%cd ../..

## 5. Direct Comparison: Scalar CPU vs. CUDA GPU
Runs both backends on identical input (`N=1000`, 100 steps) and computes:
- **Numerical error** (max & RMS position/velocity delta)
- **Exact Speed-Up factor** ($S = T_{CPU} / T_{GPU}$)

In [ ]:
!mkdir -p backends/cuda/out backends/serial/out bench/results/cpp-serial bench/results/cuda/colab

# Run Scalar CPU baseline
!backends/serial/nbody_serial --input data/ic/ic_N1000_s42.csv --steps 100 --final-state backends/serial/out/serial_1000.csv --output-json bench/results/cpp-serial/serial_1000.json

# Run CUDA Tiled GPU
!backends/cuda/nbody_cuda --input data/ic/ic_N1000_s42.csv --steps 100 --variant tiled-f64 --final-state backends/cuda/out/cuda_1000.csv --output-json bench/results/cuda/colab/cuda_1000.json

# Compare accuracy and performance
!python bench/compare.py --serial-json bench/results/cpp-serial/serial_1000.json --gpu-json bench/results/cuda/colab/cuda_1000.json --serial-csv backends/serial/out/serial_1000.csv --gpu-csv backends/cuda/out/cuda_1000.csv

## 6. Milestone 1 Validation Gate
Tests `naive`, `tiled`, `copystep` in f64 and tests non-divisible block sizes (e.g. `B=96`).

In [ ]:
# Verification test at N=100 and N=1000
for n in [100, 1000]:
    for k in ["naive", "tiled", "copystep"]:
        cmd = f"backends/cuda/nbody_cuda --input data/ic/ic_N{n}_s42.csv --steps 10 --variant {k}-f64 --final-state backends/cuda/out/final_{k}_{n}.csv"
        !{cmd}

# Test block size not dividing N (B=96 at N=1000)
!backends/cuda/nbody_cuda --input data/ic/ic_N1000_s42.csv --steps 10 --variant tiled-f64 --block-size 96 --final-state backends/cuda/out/final_tiled_b96_1000.csv

# Test FP32 run
!backends/cuda/nbody_cuda --input data/ic/ic_N1000_s42.csv --steps 10 --variant tiled-f32 --final-state backends/cuda/out/f32_1000.csv

print("All validation runs finished successfully!")

## 7. Milestone 2: Automated Benchmark Sweeps
Runs full matrix across $N$, block sizes, precision, and transfer costs.

In [ ]:
!python backends/cuda/sweep_cuda.py --machine shadhin-colab-t4 --steps 100 --repeats 3

## 8. Package & Download Results

In [ ]:
!zip -qr cuda_results.zip bench/results
from google.colab import files
files.download("cuda_results.zip")
print("Results zip downloaded!")